In [3]:
import dataretrieval.waterdata as wd
import pandas as pd

# USGS NWIS Hello World
# Fetches daily streamflow data for a single gauging station in Iowa
#
# Site 05482000 = Raccoon River at Des Moines, IA
# Parameter code 00060 = discharge (streamflow) in cubic feet per second
# Time format is ISO 8601: "start/end" or "PnD" for last n days

df, metadata = wd.get_daily(
    monitoring_location_id="USGS-05482300",
    parameter_code="00600",
    time="2020-01-01T00:00:00Z/2020-12-31T00:00:00Z",
)

print("Columns:", df.columns.tolist())
print(f"\nRows returned: {len(df)}")
print(f"\nMetadata: {metadata}")

# --- Notes ---
# monitoring_location_id format is always "USGS-{site_number}"
# To find site numbers for a state or region, use:
#   wd.get_monitoring_locations(state_name="Iowa", site_type="Stream")
#
# Common parameter codes:
#   00060 — discharge / streamflow (cfs)
#   00065 — gauge height (ft)
#   00010 — water temperature (C)
#   00400 — pH
#   00300 — dissolved oxygen (mg/L)
#
# Time can also be specified as a duration, e.g.:
#   time="P30D"  ->  last 30 days
#   time="P1Y"   ->  last year

Retrieving: daily · 1 page


Columns: ['geometry', 'time_series_id', 'monitoring_location_id', 'parameter_code', 'statistic_id', 'time', 'value', 'unit_of_measure', 'approval_status', 'qualifier', 'last_modified', 'daily_id']

Rows returned: 0

Metadata: BaseMetadata(url=https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily/items?monitoring_location_id=USGS-05482300&parameter_code=00600&time=2020-01-01T00%3A00%3A00Z%2F2020-12-31T00%3A00%3A00Z&skipGeometry=False&limit=50000)


In [7]:
from dataretrieval import waterdata

# 1. Define the state name (Note: The new API often uses full state names rather than abbreviations)
state = "Iowa"

# 2. Define the USGS NWIS Parameter Codes for nitrogen
nitrogen_pcodes = [
    "00600", "00631", "00630", "00618", 
    "00620", "00608", "00610", "00625"
]

# 3. Query the time series metadata to find which records contain nitrogen data in Iowa
timeseries_df, ts_metadata = waterdata.get_time_series_metadata(
    state_name=state,
    parameter_code=nitrogen_pcodes,
    skip_geometry=True # Speeds up the query if we don't need geometries in this step
)

# 4. Extract the unique monitoring location IDs that possess this data
monitoring_location_ids = timeseries_df['monitoring_location_id'].unique().tolist()

# 5. Query the monitoring locations endpoint to get the site attributes (lat/lon, names, etc.)
sites_df, site_metadata = waterdata.get_monitoring_locations(
    monitoring_location_id=monitoring_location_ids
)

print(f"Found {len(monitoring_location_ids)} unique sites in Iowa with nitrogen data.")
print("Sample of Location IDs:", monitoring_location_ids[:10])

Retrieving: time-series-metadata · 1 page
Retrieving: monitoring-locations · 30 pages · 1,500,000 rows


KeyboardInterrupt: 

In [10]:
from dataretrieval import waterdata

state = "Iowa"
nitrogen_pcodes = ["00600", "00631", "00630", "00618", "00620", "00608", "00610", "00625"]

# THE FIX: Join the list into a comma-separated string
pcodes_str = ",".join(nitrogen_pcodes)

# Query using the correctly formatted string
timeseries_df, ts_metadata = waterdata.get_time_series_metadata(
    state_name=state,
    parameter_code=pcodes_str, 
    skip_geometry=True 
)

# Extract the unique IDs
monitoring_location_ids = timeseries_df['monitoring_location_id'].unique().tolist()
print(f"Total unique sites: {len(monitoring_location_ids)}")

Retrieving: time-series-metadata · 1 page


Total unique sites: 0


In [11]:
from dataretrieval import waterdata

state = "Iowa"

# We added '99133' to capture the continuous in-situ nitrate sensors
nitrogen_pcodes = ["99133", "00600", "00631", "00630", "00618", "00620", "00608", "00610", "00625"]

# Join the list into a comma-separated string
pcodes_str = ",".join(nitrogen_pcodes)

# Query the time-series catalog
timeseries_df, ts_metadata = waterdata.get_time_series_metadata(
    state_name=state,
    parameter_code=pcodes_str, 
    skip_geometry=True 
)

# Extract the unique IDs
monitoring_location_ids = timeseries_df['monitoring_location_id'].unique().tolist()
print(f"Total unique continuous sites: {len(monitoring_location_ids)}")

Retrieving: time-series-metadata · 1 page · 57 rows


Total unique continuous sites: 27


In [13]:
# 1. Select the modern columns: ID, Name, and the Geometry point
site_locations = sites_df[['monitoring_location_id', 'monitoring_location_name', 'geometry']].copy()

# 2. Extract the latitude (Y) and longitude (X) from the GeoPandas geometry column 
# (Optional, but highly recommended if you are feeding this into a standard ML model)
site_locations['latitude'] = site_locations.geometry.y
site_locations['longitude'] = site_locations.geometry.x

# 3. If you want to drop the geometry column entirely afterward for standard tabular modeling:
# site_locations = site_locations.drop(columns=['geometry'])

print(site_locations.head())

  monitoring_location_id                           monitoring_location_name  \
0          USGS-05412500                         Turkey River at Garber, IA   
1          USGS-05418110  NF Maquoketa River below Bear Cr at Dyersville...   
2          USGS-05418400         North Fork Maquoketa River near Fulton, IA   
3          USGS-05418720              Maquoketa River near Green Island, IA   
4          USGS-05420500                   Mississippi River at Clinton, IA   

                     geometry   latitude  longitude  
0   POINT (-91.2618 42.73999)  42.739988 -91.261799  
1  POINT (-91.12456 42.47033)  42.470333 -91.124556  
2  POINT (-90.72933 42.16433)  42.164333 -90.729333  
3  POINT (-90.33508 42.16386)  42.163861 -90.335083  
4  POINT (-90.25207 41.78059)  41.780586 -90.252073  


In [21]:
from dataretrieval import waterdata

# The time interval uses the same ISO 8601 format (YYYY-MM-DD/YYYY-MM-DD)
time_interval = "2012-01-01/2013-01-01"

# Fetch the high-frequency continuous values
continuous_data_df, continuous_metadata = waterdata.get_continuous(
    monitoring_location_id=monitoring_location_ids, 
    parameter_code="99133",
    time=time_interval
)

print("Unaggregated data downloaded successfully!")
print(continuous_data_df.head())

Retrieving: continuous · 5 pages · 226,799 rows


Unaggregated data downloaded successfully!
                     time_series_id monitoring_location_id parameter_code  \
0  bb9535c0cf584d59b76cfb9eaf267c4d          USGS-05483600          99133   
1  bb9535c0cf584d59b76cfb9eaf267c4d          USGS-05483600          99133   
2  bb9535c0cf584d59b76cfb9eaf267c4d          USGS-05483600          99133   
3  bb9535c0cf584d59b76cfb9eaf267c4d          USGS-05483600          99133   
4  bb9535c0cf584d59b76cfb9eaf267c4d          USGS-05483600          99133   

  statistic_id                      time  value unit_of_measure  \
0        00011 2012-01-01 00:00:00+00:00   0.67            mg/l   
1        00011 2012-01-01 00:15:00+00:00   0.67            mg/l   
2        00011 2012-01-01 00:30:00+00:00   0.67            mg/l   
3        00011 2012-01-01 00:45:00+00:00   0.67            mg/l   
4        00011 2012-01-01 01:00:00+00:00   0.67            mg/l   

  approval_status qualifier                    last_modified  \
0        Approved      None

In [22]:
continuous_data_df.to_csv("test.csv")